# 16. 검색 × 생성 모델 비교 (v2.1): 최종 챗봇 조합 고르기

15에서 7B는 SFT를 해도 원본(.736)을 넘지 못했습니다 → **같은 문단에서 7B가 낼 수 있는 한계**. 남은 두 레버를 같은 심판(14)으로 비교합니다.

| 단계 | 비교 | 고정 |
|---|---|---|
| ① 검색 | S0 MiniLM 상위 5 (08) / E5 앙상블 상위 5 / E10 앙상블 상위 10 | 생성 Qwen2.5-7B 원본 |
| ② 생성 | Qwen2.5-7B / **Qwen2.5-14B** / **Qwen3-8B** / **Qwen3-14B** (thinking 끔) | ①에서 고른 검색 |
| ③ 교차 심판 | 최종 후보들을 **Phi-4 (14B, 다른 계열)**로도 채점 → Qwen 심판이 Qwen 답을 편애하는지 확인 | |
| ④ 거절 지표 수정 | 시험 '답 없음' 100개를 심판이 "문단에 답 있음/없음"으로 나눔 → **진짜 답 없는 질문**의 거절률 | |

- 앙상블 = bge-reranker-large + v2-m3 표준화 평균 (11 검증 선택), 후보·점수는 11 결과(`rerank2/`) 재사용
- 거절 규칙은 08과 같음 (MiniLM 최고점 < 4.92 → LLM 안 부르고 거절)
- **고르기는 검증 200 (답 있음)**, 시험 650은 보고용. 심판 캐시는 14·15와 공유
- 단계별 저장 → 끊겨도 모두 실행하면 이어서. 먼저 `QUICK = True`로 점검

## 1. 설치 + 경로

In [ ]:
!pip -q install -U peft faiss-cpu sentence-transformers
!pip -q uninstall -y torchao   # 최신 peft가 Colab 기본 torchao(0.10)와 충돌해서 제거 (14·15에서 확인한 조합)

In [ ]:
QUICK = False   # True: 질문 수십 개로 점검 (결과는 gen_compare/quick) / False: 본 실행

import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR   = '/content/drive/MyDrive/rag_msmarco/health'
RAG_DIR    = f'{SAVE_DIR}/rag'          # 08 시험 답변
R2_DIR     = f'{SAVE_DIR}/rerank2'      # 11 후보·재정렬 점수
JUDGE_DIR  = f'{SAVE_DIR}/judge_eval'   # 14 검증 답변 + 심판 캐시
OUT_DIR    = f'{SAVE_DIR}/gen_compare' + ('/quick' if QUICK else '')
REPORT_DIR = '/content/drive/MyDrive/RAG/2_results/generation_v21'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(REPORT_DIR, exist_ok=True)
print('QUICK =', QUICK, '→', OUT_DIR)

## 2. 설정 + 공통 함수

In [ ]:
import re, json, gzip, time, random, string, hashlib, gc
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch, transformers, peft

SEED      = 42
THRESH    = 4.921875
ENSEMBLE  = ['bge-reranker-large', 'bge-reranker-v2-m3']
BASE_RR   = 'ms-marco-MiniLM-L-6-v2'
BASE_GEN  = 'Qwen/Qwen2.5-7B-Instruct'
GENS      = ['Qwen/Qwen2.5-14B-Instruct', 'Qwen/Qwen3-8B', 'Qwen/Qwen3-14B']
JUDGE_LLM = 'Qwen/Qwen2.5-14B-Instruct'
JUDGE2    = 'microsoft/phi-4'
GEN_BS    = 16
N_TEST_A, N_TEST_N, N_TEST_O, N_VAL = (20, 5, 5, 12) if QUICK else (None, None, None, None)
DEMO = ['why does my face turn red and itchy after drinking alcohol',
        'what are the symptoms of low vitamin d',
        'can ibuprofen cause stomach bleeding',
        'how long does the flu last',
        'what is a normal resting heart rate for adults']

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (A100 권장).'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')
print(f'torch {torch.__version__} / transformers {transformers.__version__} / peft {peft.__version__}')

def load_jsonl(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def toks(s):
    s = s.lower(); s = re.sub(r'\[\d+\]', ' ', s)
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref); c = sum((Counter(p) & Counter(r)).values())
    if not p or not r or c == 0:
        return 0.0
    pr, rc = c / len(p), c / len(r)
    return 2 * pr * rc / (pr + rc)

REFUSAL = "Sorry, I can't answer that from the medical documents I have."
def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

SYS_RAG = f"""You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences."""

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    print(f'GPU 사용 {torch.cuda.memory_allocated() / 1e9:.1f}GB')

corpus = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
texts  = [c['text'] for c in corpus]
def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h
sites  = [site_of(c['url']) for c in corpus]
norm_text = lambda t: ' '.join(t.lower().split())
docs_text = lambda pids: '\n'.join(f'[{i}] ({sites[p]}) {texts[p]}' for i, p in enumerate(pids, 1))
def rag_messages(q, pids):
    return [{'role': 'system', 'content': SYS_RAG}, {'role': 'user', 'content': f'Documents:\n{docs_text(pids)}\n\nQuestion: {q}'}]
print(f'문단 {len(texts):,}')

## 3. 질문 + 문단 (11 후보·점수로 S0/E5/E10)

In [ ]:
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
C  = np.load(f'{R2_DIR}/candidates.npz')
SC = {n: np.load(f'{R2_DIR}/scores_{n}.npz') for n in ENSEMBLE + [BASE_RR]}
zs = lambda m: (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-6)

def pick(order_scores, ids, n):
    docs, seen = [], set()
    for j in np.argsort(-order_scores):
        p = int(ids[j]); key = norm_text(texts[p])
        if key in seen:
            continue
        seen.add(key); docs.append(p)
        if len(docs) == n:
            break
    return docs

groups = {'test':  [q for q in eval_set if q['domain'] == 'health' and q.get('answerable', True) and q.get('gold_pids')],
          'noans': [q for q in eval_set if q['domain'] == 'health' and not q.get('answerable', True)],
          'other': [q for q in eval_set if q['domain'] != 'health'],
          'val':   val_set}
info = {}
for g, qs in groups.items():
    ids = C[f'{g}_ids']
    assert len(ids) == len(qs), f'{g}: 11 후보 수 {len(ids)} ≠ 질문 수 {len(qs)}'
    ens = (zs(SC[ENSEMBLE[0]][g]) + zs(SC[ENSEMBLE[1]][g])) / 2
    base = SC[BASE_RR][g]
    for i, q in enumerate(qs):
        info[(g == 'val', str(q['qid']))] = {'top_minilm': float(base[i].max()), 'M5': pick(base[i], ids[i], 5),
                                              'E5': pick(ens[i], ids[i], 5), 'E10': pick(ens[i], ids[i], 10)}

# 시험 650: 08 답변 (S0 = 'RAG 개선', 원본 7B + MiniLM 상위 5)
evq = {str(q['qid']): q for q in eval_set}
test_rows = []
for r in load_jsonl(f'{RAG_DIR}/eval_answers.jsonl'):
    q = evq[str(r['qid'])]; inf = info[(False, str(r['qid']))]
    g = 'A' if r['domain'] == 'health' and r['answerable'] else ('N' if r['domain'] == 'health' else 'O')
    test_rows.append({'qid': str(r['qid']), 'query': r['query'], 'answers': q['answers'], 'gold_pids': q.get('gold_pids') or [],
                      'group': g, 'refuse': r['rag_docs'][0]['score'] < THRESH,
                      'docs': {'S0': [d['pid'] for d in r['rag_docs']], 'E5': inf['E5'], 'E10': inf['E10']},
                      'ans': {'S0|7B': r['RAG 개선']}})
# 검증 200 (답 있음): 14 답변 (S0 = 'RAG 개선')
v14 = {str(r['qid']): r for r in load_jsonl(f'{JUDGE_DIR}/val_answers.jsonl') if r['group'] == 'A'}
val_rows = []
for q in val_set:
    r = v14.get(str(q['qid'])); inf = info[(True, str(q['qid']))]
    if r is None:
        continue
    val_rows.append({'qid': str(q['qid']), 'query': q['query'], 'answers': r['answers'], 'gold_pids': q.get('gold_pids') or [],
                     'group': 'A', 'refuse': r['rag_docs'][0]['score'] < THRESH,
                     'docs': {'S0': [d['pid'] for d in r['rag_docs']], 'E5': inf['E5'], 'E10': inf['E10']},
                     'ans': {'S0|7B': r['RAG 개선']}})
if QUICK:
    by = lambda g: [r for r in test_rows if r['group'] == g]
    test_rows = by('A')[:N_TEST_A] + by('N')[:N_TEST_N] + by('O')[:N_TEST_O]
    val_rows = val_rows[:N_VAL]
print(f'시험 {len(test_rows)} {Counter(r["group"] for r in test_rows)} / 검증 {len(val_rows)}')
for k in ('S0', 'E5', 'E10'):
    for name, rows in [('시험', test_rows), ('검증', val_rows)]:
        A = [r for r in rows if r['group'] == 'A']
        print(f'  {k} {name} 근거 R@N {np.mean([bool(set(r["gold_pids"]) & set(r["docs"][k])) for r in A]):.3f}', end='')
    print()

ANS_PATH = f'{OUT_DIR}/answers.jsonl'
def save_answers():
    save_jsonl([{'set': s, 'qid': r['qid'], 'ans': r['ans']} for s, rows in [('test', test_rows), ('val', val_rows)] for r in rows], ANS_PATH)
if os.path.exists(ANS_PATH):
    saved = {(d['set'], d['qid']): d['ans'] for d in load_jsonl(ANS_PATH)}
    for s, rows in [('test', test_rows), ('val', val_rows)]:
        for r in rows:
            r['ans'].update(saved.get((s, r['qid']), {}))
    print('저장된 답변 불러옴:', sorted({k for r in test_rows for k in r['ans']}))

## 4. 생성 함수 + 심판 함수 (14와 같은 프롬프트·캐시)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_gen(name):
    tok = AutoTokenizer.from_pretrained(name, padding_side='left')
    llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map='cuda').eval()
    return tok, llm

@torch.inference_mode()
def generate(tok, llm, msgs_list, bs=GEN_BS, max_new=256):
    outs = []
    for i in range(0, len(msgs_list), bs):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False)
                   for m in msgs_list[i:i + bs]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id or tok.eos_token_id)
        dec = tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        outs += [re.sub(r'<think>.*?</think>', '', o, flags=re.S).strip() for o in dec]
    return outs

def run_system(tok, llm, key, docs_key):
    # key = '검색|모델' 시스템 답변을 시험·검증 모두 생성 (거절 규칙 08과 같음)
    for rows in (test_rows, val_rows):
        todo = [r for r in rows if not r['refuse'] and key not in r['ans']]
        t = time.time()
        outs = generate(tok, llm, [rag_messages(r['query'], r['docs'][docs_key]) for r in todo])
        for r, a in zip(todo, outs):
            r['ans'][key] = a
        for r in rows:
            if r['refuse']:
                r['ans'][key] = REFUSAL
        if todo:
            print(f'  {key}: {len(todo)}개 생성 {time.time() - t:.0f}초')
    save_answers()

SYS_CORRECT = ('You grade answers to health questions. Compare the candidate answer with the reference answer(s). '
               'Reply with one digit only.\n'
               '2 = correct: states the key facts of a reference answer (different wording or extra correct detail is fine)\n'
               '1 = partially correct: some key facts, but incomplete or partly wrong\n'
               '0 = wrong, unrelated, or does not answer the question')
SYS_FAITH = ('You check whether an answer is supported by the given documents. Reply with one digit only.\n'
             '1 = every factual statement in the answer is stated in the documents\n'
             '0 = the answer contains a fact that is not in the documents, or contradicts them')
SYS_EVID = ('You check whether the given documents contain the information needed to answer a health question. '
            'Reply with one digit only.\n'
            '1 = at least one document directly answers the question\n'
            '0 = the documents do not answer the question (related topic only, or missing information)')
def correct_msgs(q, refs, ans):
    ref = '\n'.join(f'- {x}' for x in refs[:3])
    return [{'role': 'system', 'content': SYS_CORRECT},
            {'role': 'user', 'content': f'Question: {q}\nReference answer(s):\n{ref}\nCandidate answer: {ans}\n\nGrade (0, 1 or 2):'}]
def faith_msgs(dt, ans):
    return [{'role': 'system', 'content': SYS_FAITH},
            {'role': 'user', 'content': f'Documents:\n{dt}\n\nAnswer: {ans}\n\nSupported (0 or 1)?'}]
def evid_msgs(q, dt):
    return [{'role': 'system', 'content': SYS_EVID},
            {'role': 'user', 'content': f'Documents:\n{dt}\n\nQuestion: {q}\n\nDo the documents answer it (0 or 1)?'}]

class Judge:
    def __init__(self, name, cache_path, tag=''):
        self.tok = AutoTokenizer.from_pretrained(name, padding_side='left')
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map='cuda').eval()
        self.digits = [self.tok.encode(d, add_special_tokens=False)[-1] for d in '012']
        self.path, self.tag = cache_path, tag
        self.cache = {j['key']: j for j in load_jsonl(cache_path)} if os.path.exists(cache_path) else {}
    def key(self, kind, q, a):
        return hashlib.md5(f'{self.tag}{kind}|{q}|{a}'.encode()).hexdigest()
    @torch.inference_mode()
    def probs(self, msgs_list, n, bs=8):
        prompts = [self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
        order = np.argsort([len(p) for p in prompts]); out = [None] * len(prompts)
        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            enc = self.tok([prompts[i] for i in idx], return_tensors='pt', padding=True, add_special_tokens=False).to(self.llm.device)
            logits = self.llm(**enc).logits[:, -1, self.digits[:n]].float()
            for i, p in zip(idx, torch.softmax(logits, -1).cpu().numpy()):
                out[i] = p
        return out
    def score(self, kind, items):
        # 'c' (q, refs, ans) → 0~2 / 'f' (q, docs_text, ans) → 0~1 / 'e' (q, docs_text) → 0~1
        keyed = [(self.key(kind, it[0], it[-1]), it) for it in items]
        need = list({k: it for k, it in keyed if k not in self.cache}.items())
        t = time.time()
        for s in range(0, len(need), 2000):
            part = need[s:s + 2000]
            if kind == 'c':
                vals = [float(p[1] + 2 * p[2]) for p in self.probs([correct_msgs(*it) for _, it in part], 3)]
            elif kind == 'f':
                vals = [float(p[1]) for p in self.probs([faith_msgs(it[1], it[2]) for _, it in part], 2)]
            else:
                vals = [float(p[1]) for p in self.probs([evid_msgs(it[0], it[1]) for _, it in part], 2)]
            for (k, _), v in zip(part, vals):
                self.cache[k] = {'key': k, 'v': v}
            save_jsonl(list(self.cache.values()), self.path)
        if need:
            print(f'  심판{self.tag} {kind}: {len(need)}개 새로 채점 ({time.time() - t:.0f}초)')
        return [self.cache[k]['v'] for k, _ in keyed]
    def close(self):
        del self.llm; free_gpu()

def evaluate(J, rows, systems, faith=True):
    table = {}
    for key in systems:
        dk = key.split('|')[0]
        A = [r for r in rows if r['group'] == 'A']
        ansA = [r for r in A if not is_refusal(r['ans'][key])]
        cs = dict(zip([id(r) for r in ansA], J.score('c', [(r['query'], r['answers'], r['ans'][key]) for r in ansA])))
        c_all = [cs.get(id(r), 0.0) for r in A]
        g = lambda k: [r for r in rows if r['group'] == k]
        row = {'정답률 (심판 2점)': np.mean([c >= 1.5 for c in c_all]),
               '정답+부분': np.mean([c >= 0.5 for c in c_all]),
               '평균 점수 (0~2)': np.mean(c_all),
               'F1 (참고)': np.mean([max(f1(r['ans'][key], x) for x in r['answers']) for r in A]),
               '잘못 거절': np.mean([is_refusal(r['ans'][key]) for r in A])}
        if faith:
            answered = [r for r in rows if not is_refusal(r['ans'][key])]
            fs = J.score('f', [(r['query'], docs_text(r['docs'][dk]), r['ans'][key]) for r in answered])
            row['근거 충실도'] = np.mean([f >= 0.5 for f in fs]) if fs else np.nan
        if g('N'):
            row['답 없음 거절률 (MS MARCO 기준)'] = np.mean([is_refusal(r['ans'][key]) for r in g('N')])
        if g('O'):
            row['분야 밖 거절률'] = np.mean([is_refusal(r['ans'][key]) for r in g('O')])
        table[key] = row
    return pd.DataFrame(table).T.round(3)

JUDGE_PATH = f'{JUDGE_DIR}/judgments.jsonl'

## 5. ① 검색 비교: 원본 7B로 E5·E10 답변 생성 → 심판

In [ ]:
RETR = ['S0', 'E5', 'E10']
if any(f'{k}|7B' not in r['ans'] for r in test_rows + val_rows for k in RETR):
    tok, llm = load_gen(BASE_GEN)
    for k in RETR[1:]:
        run_system(tok, llm, f'{k}|7B', k)
    del llm; free_gpu()

J = Judge(JUDGE_LLM, JUDGE_PATH)
retr_val  = evaluate(J, val_rows, [f'{k}|7B' for k in RETR])
retr_test = evaluate(J, test_rows, [f'{k}|7B' for k in RETR])
print('[검증 200: 검색 비교 (7B)]'); display(retr_val)
print(f'[시험 {len(test_rows)}: 검색 비교 (7B)]'); display(retr_test)
# 검증 정답률 최고 (차이 0.01 이내면 문단이 적은 쪽)
acc = retr_val['정답률 (심판 2점)']
best_k = max(RETR, key=lambda k: (round(acc[f'{k}|7B'] / 0.01), -len(test_rows[0]['docs'][k])))
BEST_RETR = best_k
print('→ 고른 검색:', BEST_RETR)
J.close(); del J

## 6. ② 생성 모델 비교 (고른 검색에서)

In [ ]:
short = lambda n: n.split('/')[-1].replace('-Instruct', '')
for name in GENS:
    key = f'{BEST_RETR}|{short(name)}'
    if any(key not in r['ans'] for r in test_rows + val_rows):
        t = time.time()
        tok, llm = load_gen(name)
        run_system(tok, llm, key, BEST_RETR)
        del llm; free_gpu()
        print(f'{short(name)} 끝 ({(time.time() - t) / 60:.0f}분)')
GEN_KEYS = [f'{BEST_RETR}|7B'] + [f'{BEST_RETR}|{short(n)}' for n in GENS]
if BEST_RETR != 'S0':
    GEN_KEYS = ['S0|7B'] + GEN_KEYS

In [ ]:
J = Judge(JUDGE_LLM, JUDGE_PATH)
gen_val  = evaluate(J, val_rows, GEN_KEYS)
gen_test = evaluate(J, test_rows, GEN_KEYS)
print('[검증 200: 생성 모델 비교]'); display(gen_val)
print(f'[시험 {len(test_rows)}: 생성 모델 비교]'); display(gen_test)
BEST = gen_val['정답률 (심판 2점)'].idxmax()
print('→ 검증 기준 최고:', BEST)

# 질문별 정답률 차이 부트스트랩 (최고 - 08 S0|7B), 시험
A = [r for r in test_rows if r['group'] == 'A']
def corr(r, key):
    return 0.0 if is_refusal(r['ans'][key]) else J.cache[J.key('c', r['query'], r['ans'][key])]['v']
def boot(key, base='S0|7B'):
    d = np.array([(corr(r, key) >= 1.5) - (corr(r, base) >= 1.5) for r in A], dtype=float)
    bs = np.random.default_rng(SEED).choice(d, (2000, len(d))).mean(1)
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
ci_tab = pd.DataFrame({k: dict(zip(['차이', '95% CI 하한', '95% CI 상한'], boot(k))) for k in GEN_KEYS if k != 'S0|7B'}).T.round(3)
print('[시험 정답률 차이 vs 08 (S0|7B)]'); display(ci_tab)

## 7. ④ 거절 지표 수정: 시험 '답 없음' 100개를 심판이 "문단에 답 있음/없음"으로 나눔

MS MARCO '답 없음'은 원래 문단 10개 기준이라, 46만 DB에서 다시 검색하면 답이 들어오기도 합니다 (15에서 465개 중 261개).
- **진짜 답 없음** (P(문단에 답 있음) < 0.2): 거절해야 정답 → 거절률
- **사실 답 있음** (≥ 0.8): 답해도 됨 → 답변률·근거 충실도

In [ ]:
N = [r for r in test_rows if r['group'] == 'N']
ref_rows = {}
for key in GEN_KEYS:
    dk = key.split('|')[0]
    ev = J.score('e', [(r['query'], docs_text(r['docs'][dk])) for r in N])
    no = [r for r, e in zip(N, ev) if e < 0.2]; yes = [r for r, e in zip(N, ev) if e >= 0.8]
    ans_yes = [r for r in yes if not is_refusal(r['ans'][key])]
    fs = J.score('f', [(r['query'], docs_text(r['docs'][dk]), r['ans'][key]) for r in ans_yes])
    ref_rows[key] = {'진짜 답 없음 (개)': len(no), '→ 거절률': np.mean([is_refusal(r['ans'][key]) for r in no]) if no else np.nan,
                     '사실 답 있음 (개)': len(yes), '→ 답변률': len(ans_yes) / len(yes) if yes else np.nan,
                     '→ 근거 충실도': np.mean([f >= 0.5 for f in fs]) if fs else np.nan}
ref_tab = pd.DataFrame(ref_rows).T.round(3)
print('[시험 답 없음 100: 심판 기준으로 다시 본 거절]'); display(ref_tab)
J.close(); del J

## 8. ③ 교차 심판 (Phi-4): Qwen 심판이 Qwen 답을 편애하지 않는지

정답 점수만, 시험·검증 답 있음 질문에 대해. 순위가 같으면 결론을 믿을 수 있습니다.

In [ ]:
J2 = Judge(JUDGE2, f'{OUT_DIR}/judgments_phi4.jsonl', tag='phi4|')
x_val  = evaluate(J2, val_rows, GEN_KEYS, faith=False)[['정답률 (심판 2점)', '평균 점수 (0~2)']]
x_test = evaluate(J2, test_rows, GEN_KEYS, faith=False)[['정답률 (심판 2점)', '평균 점수 (0~2)']]
cross = pd.DataFrame({'Qwen 심판 검증': gen_val['정답률 (심판 2점)'], 'Phi-4 심판 검증': x_val['정답률 (심판 2점)'],
                      'Qwen 심판 시험': gen_test['정답률 (심판 2점)'], 'Phi-4 심판 시험': x_test['정답률 (심판 2점)']}).round(3)
print('[정답률: 심판 2개 비교]'); display(cross)
same_rank = list(cross['Qwen 심판 검증'].rank()) == list(cross['Phi-4 심판 검증'].rank())
print('검증 순위 일치:', same_rank, '/ Phi-4 검증 최고:', cross['Phi-4 심판 검증'].idxmax())
J2.close(); del J2

## 9. 데모 질문 (최고 조합 vs 08)

In [ ]:
demo = []
if True:   # 데모 질문은 검색부터 새로 (MiniLM 거절 점수 + 앙상블)
    import faiss
    from sentence_transformers import SentenceTransformer, CrossEncoder
    index = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
    emb = SentenceTransformer('BAAI/bge-base-en-v1.5', device='cuda').half()
    qe = emb.encode(['Represent this sentence for searching relevant passages: ' + q for q in DEMO],
                    normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    _, ids = index.search(qe, 100)
    pairs = [(q, texts[p]) for q, c in zip(DEMO, ids.tolist()) for p in c]
    mini = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=256, device='cuda').predict(pairs, batch_size=256).reshape(len(DEMO), 100)
    ens = 0
    for m in ['BAAI/bge-reranker-large', 'BAAI/bge-reranker-v2-m3']:
        ce = CrossEncoder(m, max_length=512, device='cuda'); ce.model.half()
        ens = ens + zs(ce.predict(pairs, batch_size=64).reshape(len(DEMO), 100)) / 2
        del ce
    del emb, index; free_gpu()
    for i, q in enumerate(DEMO):
        demo.append({'query': q, 'refuse': mini[i].max() < THRESH,
                     'docs': {'S0': pick(mini[i], ids[i], 5), 'E5': pick(ens[i], ids[i], 5), 'E10': pick(ens[i], ids[i], 10)}})
for key in dict.fromkeys(['S0|7B', BEST]):
    dk, mk = key.split('|')
    name = BASE_GEN if mk == '7B' else next(n for n in GENS if short(n) == mk)
    tok, llm = load_gen(name)
    outs = generate(tok, llm, [rag_messages(d['query'], d['docs'][dk]) for d in demo])
    for d, a in zip(demo, outs):
        d[key] = REFUSAL if d['refuse'] else a
    del llm; free_gpu()
for d in demo:
    print(f"Q: {d['query']}\n  08 (S0|7B): {d['S0|7B'][:220]}\n  {BEST}: {d[BEST][:220]}")

## 10. 저장

In [ ]:
stamp = time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))
demo_md = '\n'.join(f"- **{d['query']}**\n  - 08 (S0|7B): {d['S0|7B']}\n  - {BEST}: {d[BEST]}" for d in demo)
summary = '\n'.join([
    f'# 16 검색 × 생성 모델 비교 (v2.1){" [QUICK 점검]" if QUICK else ""} ({stamp} KST)', '',
    '- S0 = MiniLM 상위 5 (08), E5/E10 = 앙상블(bge-reranker-large + v2-m3) 상위 5/10. 거절은 08과 같음 (MiniLM 최고점 < 4.92)',
    f'- 심판: {JUDGE_LLM} (14와 같음), 교차 심판 {JUDGE2}. 고르기는 검증 {len(val_rows)}, 시험 {len(test_rows)}은 보고용',
    f'- **고른 검색: {BEST_RETR} / 검증 기준 최고 조합: {BEST}** / Phi-4 검증 순위 일치: {same_rank}', '',
    '## ① 검색 비교 (생성 7B)', '', '검증', '', retr_val.to_markdown(), '', '시험', '', retr_test.to_markdown(), '',
    '## ② 생성 모델 비교', '', '검증', '', gen_val.to_markdown(), '', '시험', '', gen_test.to_markdown(), '',
    '시험 정답률 차이 (vs 08 S0|7B, 95% 부트스트랩)', '', ci_tab.to_markdown(), '',
    '## ③ 교차 심판 (정답률)', '', cross.to_markdown(), '',
    '## ④ 시험 답 없음 100: 심판 기준 거절', '', ref_tab.to_markdown(), '',
    '## 데모', '', demo_md,
])
paths = [f'{OUT_DIR}/results_summary.md'] + ([] if QUICK else [f'{REPORT_DIR}/16_gen_compare_summary.md'])
for path in paths:
    with open(path, 'w', encoding='utf-8') as f:
        f.write(summary)
json.dump({'retrieval': BEST_RETR, 'best': BEST}, open(f'{OUT_DIR}/best_config.json', 'w'), ensure_ascii=False)
print(summary)

## 11. 드라이브 저장 마무리 + 런타임 종료 (QUICK 점검일 땐 런타임 유지)

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
if not QUICK:
    runtime.unassign()